# ♟️ Chess Piece Detection and Classification using YOLOv8 and OpenCV

This notebook demonstrates a complete end-to-end pipeline for detecting and classifying chess pieces from board images using **YOLOv8** (Ultralytics) and **OpenCV**.

### Pipeline Overview
1. **Environment setup** — install and import all dependencies
2. **Dataset download** — load the public Roboflow chess detection dataset
3. **Model training** — fine-tune YOLOv8n on the chess dataset
4. **Inference & visualization** — detect pieces with annotated bounding boxes
5. **Evaluation** — mAP, precision, recall, and confusion matrix
6. **Batch inference demo** — run on a set of test images

### Classes Detected
| Color | Pieces |
|-------|--------|
| ⬜ White | King, Queen, Rook, Bishop, Knight, Pawn |
| ⬛ Black | King, Queen, Rook, Bishop, Knight, Pawn |

> **Hardware note:** Training on CPU is slow. A GPU (Google Colab or local CUDA) is recommended.
> Run on [Google Colab](https://colab.research.google.com/) for free GPU access.

---
## 1. 📦 Install Dependencies

In [ ]:
# Install required packages (uncomment if running for the first time)
# !pip install ultralytics>=8.0.0 roboflow opencv-python matplotlib seaborn scikit-learn tqdm PyYAML -q

import sys
print(f'Python version: {sys.version}')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
import glob
import shutil
import yaml
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm

from ultralytics import YOLO
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

print('✅ All imports successful!')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Matplotlib settings
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

---
## 2. 📂 Dataset Download from Roboflow

We use the **Chess Pieces Detection** public dataset from Roboflow Universe.
- Dataset: [roboflow.com/chess-pieces](https://universe.roboflow.com/chess-pieces-nemtd/chess-pieces-detection)
- 12 classes: white/black × king, queen, rook, bishop, knight, pawn
- Format: YOLOv8 (images + label `.txt` files + `data.yaml`)
- License: Public (free download via Roboflow API)

> **Get a free API key** at [app.roboflow.com](https://app.roboflow.com) → Settings → API Keys  
> Replace `"YOUR_ROBOFLOW_API_KEY"` below with your actual key.

In [ ]:
from roboflow import Roboflow

# ── Configuration ──────────────────────────────────────────────────────────────
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"   # <── Replace with your key
WORKSPACE        = "chess-pieces-nemtd"       # Roboflow workspace slug
PROJECT          = "chess-pieces-detection"   # Roboflow project slug
VERSION          = 1                           # Dataset version number
DATASET_FORMAT   = "yolov8"                   # Export format
DATA_DIR         = "chess_dataset"             # Local download folder
# ───────────────────────────────────────────────────────────────────────────────

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(DATASET_FORMAT, location=DATA_DIR)

DATA_YAML = os.path.join(DATA_DIR, 'data.yaml')
print(f'\n✅ Dataset downloaded to: {os.path.abspath(DATA_DIR)}')
print(f'   data.yaml path        : {os.path.abspath(DATA_YAML)}')

In [ ]:
# ── Inspect data.yaml ─────────────────────────────────────────────────────────
with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg['names']
NUM_CLASSES = data_cfg['nc']

print('=== data.yaml contents ===')
print(yaml.dump(data_cfg, default_flow_style=False))
print(f'Number of classes : {NUM_CLASSES}')
print(f'Class names       : {CLASS_NAMES}')

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
def count_images(split_dir):
    img_dir = os.path.join(split_dir, 'images')
    if not os.path.isdir(img_dir):
        return 0
    return len(glob.glob(os.path.join(img_dir, '*.jpg')) +
               glob.glob(os.path.join(img_dir, '*.png')) +
               glob.glob(os.path.join(img_dir, '*.jpeg')))

splits = ['train', 'valid', 'test']
counts = {s: count_images(os.path.join(DATA_DIR, s)) for s in splits}

print('=== Dataset split sizes ===')
for split, n in counts.items():
    print(f'  {split:>5}: {n:>5} images')
print(f'  Total: {sum(counts.values()):>5} images')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(counts.keys(), counts.values(),
              color=['#4C72B0', '#DD8452', '#55A868'])
ax.bar_label(bars, padding=3)
ax.set_title('Images per Dataset Split')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts.values()) * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize sample training images with ground-truth boxes ──────────────────
COLORS = plt.cm.get_cmap('tab20', NUM_CLASSES)

def draw_yolo_boxes(img_path, label_path, class_names):
    """Draw YOLO-format ground truth bounding boxes on an image."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(x) for x in parts[1:]]
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            bw_px = bw * w
            bh_px = bh * h
            color = COLORS(cls_id)[:3]
            rect = patches.Rectangle(
                (x1, y1), bw_px, bh_px,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, class_names[cls_id],
                    color='white', fontsize=8, weight='bold',
                    bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor='none'))

    ax.axis('off')
    ax.set_title(os.path.basename(img_path), fontsize=10)
    return fig


train_img_dir   = os.path.join(DATA_DIR, 'train', 'images')
train_label_dir = os.path.join(DATA_DIR, 'train', 'labels')

sample_images = random.sample(
    glob.glob(os.path.join(train_img_dir, '*.jpg')) +
    glob.glob(os.path.join(train_img_dir, '*.png')),
    k=min(4, counts['train'])
)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, img_path in zip(axes.flatten(), sample_images):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(train_label_dir, stem + '.txt')
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    if os.path.exists(lbl_path):
        h, w = img.shape[:2]
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:]]
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                color = COLORS(cls_id)[:3]
                rect = patches.Rectangle(
                    (x1, y1), bw * w, bh * h,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, max(y1 - 4, 0), CLASS_NAMES[cls_id],
                        color='white', fontsize=7, weight='bold',
                        bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor='none'))
    ax.axis('off')
    ax.set_title(os.path.basename(img_path), fontsize=9)

plt.suptitle('Sample Training Images with Ground-Truth Annotations', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sample_annotations.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: sample_annotations.png')

---
## 3. 🏋️ Model Training — Fine-tune YOLOv8

We fine-tune **YOLOv8n** (nano — fastest, smallest) on the chess dataset.  
Switch to `yolov8s.pt` or `yolov8m.pt` for better accuracy at the cost of speed.

| Model | Size | mAP50 (COCO) | Speed |
|-------|------|-------------|-------|
| YOLOv8n | 3.2M | 37.3 | Fastest |
| YOLOv8s | 11.2M | 44.9 | Fast |
| YOLOv8m | 25.9M | 50.2 | Medium |

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
MODEL_WEIGHTS  = 'yolov8n.pt'   # Pretrained base weights (auto-downloaded)
EPOCHS         = 50             # Increase to 100 for best accuracy
IMG_SIZE       = 640            # Standard YOLO input size
BATCH_SIZE     = 16             # Reduce to 8 if GPU OOM
PATIENCE       = 10             # Early stopping patience
PROJECT_NAME   = 'chess_yolo'   # Run output folder
RUN_NAME       = 'train_v1'     # Sub-folder inside PROJECT_NAME
DEVICE         = 0 if torch.cuda.is_available() else 'cpu'

print(f'Training on device : {DEVICE}')
print(f'Model              : {MODEL_WEIGHTS}')
print(f'Epochs             : {EPOCHS}')
print(f'Image size         : {IMG_SIZE}')
print(f'Batch size         : {BATCH_SIZE}')

# ── Load base model ───────────────────────────────────────────────────────────
model = YOLO(MODEL_WEIGHTS)

# ── Train ─────────────────────────────────────────────────────────────────────
results = model.train(
    data       = DATA_YAML,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    patience   = PATIENCE,
    device     = DEVICE,
    project    = PROJECT_NAME,
    name       = RUN_NAME,
    plots      = True,         # Saves training plots automatically
    save       = True,
    save_period= 10,           # Save checkpoint every 10 epochs
    exist_ok   = True,
    verbose    = True,
)

# Resolve path to best weights
BEST_WEIGHTS = os.path.join(PROJECT_NAME, RUN_NAME, 'weights', 'best.pt')
print(f'\n✅ Training complete!')
print(f'   Best weights saved at: {os.path.abspath(BEST_WEIGHTS)}')

In [ ]:
# ── Plot training curves from results.csv ─────────────────────────────────────
results_csv = os.path.join(PROJECT_NAME, RUN_NAME, 'results.csv')

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    metrics_to_plot = [
        ('train/box_loss', 'val/box_loss',     'Box Loss'),
        ('train/cls_loss', 'val/cls_loss',     'Classification Loss'),
        ('metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'mAP'),
        ('metrics/precision(B)', 'metrics/recall(B)', 'Precision & Recall'),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    colors = ['#4C72B0', '#DD8452']

    for ax, (col1, col2, title) in zip(axes.flatten(), metrics_to_plot):
        epochs_x = df['epoch'] if 'epoch' in df.columns else range(len(df))
        if col1 in df.columns:
            label1 = col1.split('/')[-1].replace('(B)', '').replace('_', ' ')
            ax.plot(epochs_x, df[col1], color=colors[0], label=label1, linewidth=2)
        if col2 in df.columns:
            label2 = col2.split('/')[-1].replace('(B)', '').replace('_', ' ')
            ax.plot(epochs_x, df[col2], color=colors[1], label=label2,
                    linewidth=2, linestyle='--')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle('Training Curves', fontsize=15)
    plt.tight_layout()
    plt.savefig('training_curves.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: training_curves.png')
else:
    print(f'results.csv not found at {results_csv}. Run training first.')

---
## 4. 📊 Model Evaluation

Evaluate the trained model on the **validation set** and generate:
- **mAP50** and **mAP50-95** scores
- **Precision** and **Recall** per class
- **Confusion Matrix**

In [ ]:
# ── Load trained model ────────────────────────────────────────────────────────
chess_model = YOLO(BEST_WEIGHTS)
print(f'✅ Loaded model from: {BEST_WEIGHTS}')

# ── Run validation ────────────────────────────────────────────────────────────
val_results = chess_model.val(
    data    = DATA_YAML,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    project = PROJECT_NAME,
    name    = 'val_v1',
    exist_ok= True,
    verbose = True,
)

# ── Print summary metrics ─────────────────────────────────────────────────────
print('\n' + '='*50)
print('       VALIDATION METRICS SUMMARY')
print('='*50)
print(f'  mAP@50        : {val_results.box.map50:.4f}')
print(f'  mAP@50-95     : {val_results.box.map:.4f}')
print(f'  Precision     : {val_results.box.mp:.4f}')
print(f'  Recall        : {val_results.box.mr:.4f}')
print('='*50)

In [ ]:
# ── Per-class metrics table ───────────────────────────────────────────────────
try:
    ap_per_class = val_results.box.ap_class_index  # class indices
    ap50_vals    = val_results.box.ap50             # AP50 per class
    ap_vals      = val_results.box.ap               # AP50-95 per class

    metrics_df = pd.DataFrame({
        'Class'      : [CLASS_NAMES[i] for i in ap_per_class],
        'AP@50'      : ap50_vals,
        'AP@50-95'   : ap_vals,
    })
    metrics_df = metrics_df.sort_values('AP@50', ascending=False).reset_index(drop=True)

    print('\nPer-class Average Precision:')
    print(metrics_df.to_string(index=False, float_format='{:.4f}'.format))

    # Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col, title in zip(axes, ['AP@50', 'AP@50-95'], ['AP@50', 'AP@50-95']):
        bars = ax.barh(metrics_df['Class'], metrics_df[col],
                       color=sns.color_palette('husl', len(metrics_df)))
        ax.set_xlim(0, 1.05)
        ax.set_xlabel(col)
        ax.set_title(f'Per-class {title}')
        ax.axvline(metrics_df[col].mean(), color='red', linestyle='--',
                   linewidth=1.5, label=f'Mean: {metrics_df[col].mean():.3f}')
        ax.legend()
        ax.grid(axis='x', alpha=0.3)

    plt.suptitle('Per-class Detection Performance', fontsize=14)
    plt.tight_layout()
    plt.savefig('per_class_metrics.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: per_class_metrics.png')

except Exception as e:
    print(f'Per-class breakdown not available: {e}')
    print('Run validation first to generate per-class metrics.')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
# YOLO saves the confusion matrix plot automatically during val — display it
cm_auto_path = os.path.join(PROJECT_NAME, 'val_v1', 'confusion_matrix_normalized.png')
cm_path2     = os.path.join(PROJECT_NAME, 'val_v1', 'confusion_matrix.png')

for cm_file in [cm_auto_path, cm_path2]:
    if os.path.exists(cm_file):
        img = plt.imread(cm_file)
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title('Confusion Matrix (YOLOv8 Validation)', fontsize=14)
        plt.tight_layout()
        plt.savefig('confusion_matrix_display.png', bbox_inches='tight', dpi=150)
        plt.show()
        print(f'Confusion matrix loaded from: {cm_file}')
        print('Saved: confusion_matrix_display.png')
        break
else:
    print('Confusion matrix image not found. Verify validation ran successfully.')

---
## 5. 🔍 Inference & Visualization

Run the trained model on individual and batch test images to visualize detections.

In [ ]:
# ── Helper: display annotated prediction ──────────────────────────────────────
def predict_and_show(model, img_path, conf=0.30, iou=0.45,
                     title=None, save_path=None):
    """
    Run YOLOv8 inference on a single image and display annotated result.

    Parameters
    ----------
    model     : YOLO – trained model
    img_path  : str  – path to input image
    conf      : float – confidence threshold (0-1)
    iou       : float – NMS IoU threshold (0-1)
    title     : str  – optional plot title
    save_path : str  – optional path to save the annotated image

    Returns
    -------
    results : list[ultralytics.engine.results.Results]
    """
    preds = model.predict(source=img_path, conf=conf, iou=iou, verbose=False)
    annotated = preds[0].plot(line_width=2, font_size=10)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    n_detections = len(preds[0].boxes)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(annotated_rgb)
    ax.axis('off')
    ax.set_title(
        title or f'{os.path.basename(img_path)}  —  {n_detections} piece(s) detected',
        fontsize=12
    )
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f'Saved: {save_path}')
    plt.show()
    return preds


# ── Single-image inference ────────────────────────────────────────────────────
test_img_dir = os.path.join(DATA_DIR, 'test', 'images')
test_images  = (
    glob.glob(os.path.join(test_img_dir, '*.jpg')) +
    glob.glob(os.path.join(test_img_dir, '*.png'))
)

if test_images:
    sample = random.choice(test_images)
    preds  = predict_and_show(
        chess_model, sample,
        conf=0.30, iou=0.45,
        save_path='inference_sample.png'
    )

    # Print detection details
    boxes  = preds[0].boxes
    print(f'\nDetections in {os.path.basename(sample)}:')
    print(f'{"Class":<20} {"Confidence":>10} {"Bounding Box (xyxy)"}')
    print('-' * 70)
    for box in boxes:
        cls_id = int(box.cls.item())
        conf_v = box.conf.item()
        xyxy   = box.xyxy[0].tolist()
        print(f'{CLASS_NAMES[cls_id]:<20} {conf_v:>10.3f}  '
              f'[{xyxy[0]:.0f}, {xyxy[1]:.0f}, {xyxy[2]:.0f}, {xyxy[3]:.0f}]')
else:
    print('No test images found. Check DATA_DIR path.')

In [ ]:
# ── Batch inference — 6 test images ──────────────────────────────────────────
N_BATCH = 6
batch_samples = random.sample(test_images, k=min(N_BATCH, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Chess Piece Detection — Batch Inference Results', fontsize=15)

for ax, img_path in zip(axes.flatten(), batch_samples):
    preds     = chess_model.predict(source=img_path, conf=0.30, iou=0.45, verbose=False)
    annotated = preds[0].plot(line_width=2, font_size=8)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    n = len(preds[0].boxes)
    ax.set_title(f'{os.path.basename(img_path)}  ({n} pieces)', fontsize=9)
    ax.axis('off')

# Hide unused subplots
for ax in axes.flatten()[len(batch_samples):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig('batch_inference.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: batch_inference.png')

In [ ]:
# ── Piece-count distribution over the test set ────────────────────────────────
print('Running inference over full test set for statistics...')

class_counts = {name: 0 for name in CLASS_NAMES}
confidence_scores = []

for img_path in tqdm(test_images, desc='Test set inference'):
    preds = chess_model.predict(source=img_path, conf=0.30, iou=0.45, verbose=False)
    for box in preds[0].boxes:
        cls_id = int(box.cls.item())
        confidence_scores.append(box.conf.item())
        class_counts[CLASS_NAMES[cls_id]] += 1

# ── Plot 1: Detection count per class
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
classes, cnts  = zip(*sorted_classes)

palette = sns.color_palette('coolwarm', len(classes))
axes[0].barh(classes, cnts, color=palette)
axes[0].set_xlabel('Detection Count')
axes[0].set_title('Piece Detections per Class (Test Set)')
axes[0].grid(axis='x', alpha=0.3)

# ── Plot 2: Confidence score distribution
axes[1].hist(confidence_scores, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(np.mean(confidence_scores), color='red', linestyle='--',
                label=f'Mean: {np.mean(confidence_scores):.3f}')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Prediction Confidence Score Distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Test Set Detection Statistics', fontsize=14)
plt.tight_layout()
plt.savefig('detection_statistics.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: detection_statistics.png')

# Summary
total_detections = sum(class_counts.values())
print(f'\nTotal detections across {len(test_images)} test images: {total_detections}')
print(f'Average confidence score : {np.mean(confidence_scores):.4f}')
print(f'Median confidence score  : {np.median(confidence_scores):.4f}')

---
## 6. 💾 Export the Model

Export the trained model to **ONNX** format for deployment in other frameworks (OpenCV DNN, TensorRT, etc.).

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────
export_path = chess_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)
print(f'✅ Model exported to ONNX: {export_path}')

# ── Model info ────────────────────────────────────────────────────────────────
chess_model.info()

print('\n=== All output files ===')
output_files = [
    BEST_WEIGHTS,
    'training_curves.png',
    'per_class_metrics.png',
    'confusion_matrix_display.png',
    'sample_annotations.png',
    'inference_sample.png',
    'batch_inference.png',
    'detection_statistics.png',
]
for f in output_files:
    status = '✅' if os.path.exists(f) else '❌'
    print(f'  {status} {f}')

---
## 7. 📝 Summary & Conclusions

This notebook implemented a complete chess piece detection and classification pipeline using YOLOv8.

### Key Results
| Metric | Value |
|--------|-------|
| mAP@50 | *(see evaluation cell output)* |
| mAP@50-95 | *(see evaluation cell output)* |
| Precision | *(see evaluation cell output)* |
| Recall | *(see evaluation cell output)* |

### Why YOLO over alternatives?
- **CNN classifier** on cropped pieces: requires a separate detection step, more complex pipeline
- **Template matching / HOG**: fails on varied board angles, lighting, and piece designs
- **YOLOv8**: single-pass detection + classification, state-of-the-art accuracy, easy fine-tuning

### Possible Improvements
- Use a larger model (`yolov8s`, `yolov8m`) for better accuracy
- Apply **data augmentation** (rotation, flipping, brightness changes)
- Train for more epochs with a larger dataset
- Extend to **real-time board analysis** using a webcam feed
- Add **FEN string generation** from detected piece positions

### References
- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com)
- [Roboflow Chess Dataset](https://universe.roboflow.com/chess-pieces-nemtd/chess-pieces-detection)
- [OpenCV Documentation](https://docs.opencv.org/4.x/)